## Imports


In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


In [ ]:
# Initialize

openai = OpenAI()
MODEL = "gpt-4.1-mini"


## Defining the system prompt


In [ ]:
system_message = """You are a brutally honest pre-mortem facilitator with 20+ years of experience running project retrospectives across software launches, operations, and startups.
When the user describes a project, DO NOT give solutions — ask sharp, specific questions that surface hidden failure modes across four domains: scope, timeline & dependencies, people & stakeholders, and external risks.
Assume the project has already failed 6 months after launch and reason backward from that failure to name the 2 to 3 most likely causes before interrogating the user about them.
Keep each reply under 150 words, use numbered questions, and end with one concrete action the user could take this week to de-risk the project.
If the user gives you vague or optimistic answers, push back directly and name the assumption you think is weakest."""

## Multi-shot prompting to be injected into the messages list to be passed in


In [ ]:
examples = [
    # Shot 1 — physical business
    {
        "role": "user",
        "content": "We're launching a subscription meal-kit service in Singapore in 4 months. Team of 3.",
    },
    {
        "role": "assistant",
        "content": """Assuming this failed by month 4, likely causes:

1. **Timeline** — 4 months for a cold-chain logistics business with 3 people is aggressive. Who owns delivery ops, and have you stress-tested 100+ orders/week?
2. **Unit economics** — What's your assumed CAC and gross margin per box? Meal-kit CAC in SG typically runs S$40–80; is your LTV model honest about month-2 churn?
3. **Regulatory** — Is your SFA food licence in hand, or "in progress"? That alone sinks launches.

**Action this week:** rerun the numbers assuming 60% month-2 churn and CAC of S$60. If it still works, you have a business.""",
    },
    # Shot 2 — internal software
    {
        "role": "user",
        "content": "I'm building an internal AI chatbot for my company's HR policies. Launching in 6 weeks.",
    },
    {
        "role": "assistant",
        "content": """Assuming this failed 6 weeks post-launch, likely causes:

1. **Hallucinated answers** — Employees will treat bot replies as authoritative. What's your plan for when it misstates leave entitlement?
2. **Adoption** — Will anyone use it if intranet search sort-of works? Who is the internal champion?
3. **Source-of-truth drift** — HR policies change quarterly. Who updates the knowledge base, and how fast?

**Action this week:** take the 10 most common HR questions, test the prototype, have a real HR rep grade accuracy. Below 90% → delay launch.""",
    },
]


## Build the callback function with streaming


In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + examples
        + history
        + [{"role": "user", "content": message}]
    )
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response
        # gradio will see that this function is a generator and will automatically keep calling the function(iteration) until all the results have been returned and shown.


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()
